# W08a Colab A · 표현·복원·검색

생성형 인공지능 · 2026년 10월 19일(월) · 동덕여자대학교 데이터 사이언스 전공 · 유원상 교수

이 노트북은 첫 셀부터 **CPU로 독립 실행**합니다. 다른 노트북이나 지난 실습의 변수가 필요 없습니다. GPU·유료 API·개인 파일 업로드·대형 모델 신규 다운로드가 필수가 아닙니다.

파일 → Drive에 사본 저장 후 사용하세요. 패키지 모듈을 이미 불러왔다면 설치 후 런타임을 재시작합니다. 오류를 숨기거나 가짜 실행 결과로 바꾸지 않습니다.

설명용 작은 수치와 기존 실제 SD 실행 기록을 구별합니다. 기존 기록을 읽었다고 지금 새 모델을 실행했다고 말하지 않습니다. 질문의 미완성 코드는 Markdown으로 제시하고, 본인의 풀이 변수는 `None`으로 남깁니다. **전체 실행 통과는 문제를 모두 풀었다는 뜻이 아닙니다.**

[학생 요약과 문제](https://github.com/lunalab-ai/genAI/tree/2026-fall-w08a-r1/course/notion/w08a) · [전체 공통 API](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/W08A-API.md)

## 함수 사용 전: 입력·출력·부작용

목록의 한 원소는 픽셀 또는 특징 좌표입니다. 같은 길이끼리 연산하고 `zip(a,b)`는 같은 위치의 값을 짝지습니다. `sum`은 합, `len`은 개수이며, `math.sqrt`는 제곱근, `math.exp`는 자연지수, `math.log`는 자연로그입니다. 행렬의 한 행이 후보 한 개라는 약속을 유지합니다.

표준 문법: [목록·반복](https://docs.python.org/3.13/tutorial/datastructures.html), [math 함수](https://docs.python.org/3.13/library/math.html). 아래 수업 함수는 설치한 고정 태그의 실제 정의 줄에 연결됩니다.


### [reconstruction_stats](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_core.py#L34)

한 이미지의 같은 길이 픽셀 두 목록 → residual, SSE, MSE, pixel_count.

residual은 prediction-target, SSE는 제곱합, MSE는 픽셀 수로 나눈 값.
예: [0,1]과[.2,.8] → SSE=.08, MSE=.04. 학습·클리핑 없음.
실제 영상의 측정값인지 설명용 입력인지 호출자가 표시해야 한다.

### [vae_quantities](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_core.py#L47)

잠재 좌표 d개의 평균·로그분산·고정 잡음 → variance/std/z/KL.

z=mu+exp(logvar/2)*epsilon; KL은 좌표합, 표준정규 prior 기준.
한 이미지의 설명용 계산이며 신경망·gradient·난수 추출 없음.
예: mu=[0], logvar=[0], epsilon=[.2] → z=[.2], KL=0.
epsilon은 관찰하려는 표준정규 잡음의 실현값을 직접 전달한다.

### [cosine_search](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_core.py#L70)

질의 d개·후보 N×d → cosine/relative_score/order/unit_rows.

각 행을 L2 정규화; order는 높은 점수 순 인덱스이며 동률은 원래 순서.
scale>0(기본1)은 softmax 점수 배율. 상대 점수는 정답 확률이 아니다.
예: [1,0], [[1,0],[0,1]] → cosine=[1,0], order=[0,1].
0벡터/비유한값/축 불일치는 오류. CLIP 인코더를 실행하지 않는다.

### [precision_at_k](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_core.py#L95)

서로 다른 검색 ID·미리 정한 관련 ID 집합·k → 상위 k의 관련 비율.

k는 양의 정수, 기본3. 결과가 k보다 적으면 빈 순위는 오답으로 센다.
예: ['A','B','C'], {'B'}, 2 → .5. 동률 정렬은 검색 단계에서 결정.
관련성 판단이나 모델 학습을 수행하지 않는다.

### [representation_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_app.py#L27)

beta·첫 잡음값 → 픽셀표/잠재표/설명/그림(한 이미지 예시).

beta>=0, epsilon_first는 유한값. 고정 복원 SSE+beta*KL을 다시 계산하며
beta 변경 후 재학습한 결과가 아니다. 표의 MSE/SSE 분모와 z를 함께 읽는다.
반환 Figure는 사용 후 close. summary_fn(record)->str는 Q60 연결 지점.

### [search_view](https://github.com/lunalab-ai/genAI/blob/2026-fall-w08a-r1/src/luna_genai/review2_app.py#L57)

추가후보 여부·양수배율·k(1..4) → 순위표/설명/그림.

질의[2,1], 후보A[1,0]/B[0,1]/C[1,1]/추가D[-1,0]는 설명용 벡터다.
관련 ID는 검색 전 {A,C}로 고정. 후보를 추가해도 기존 cosine은 같지만
softmax 분모는 달라진다. 실제 CLIP 임베딩/정답 확률이라고 표시하지 않는다.


In [ ]:
import os, sys, subprocess, math
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
IN_COLAB = 'google.colab' in sys.modules
COURSE_REF = '2026-fall-w08a-r1'
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        f'luna-genai @ git+https://github.com/lunalab-ai/genAI.git@{COURSE_REF}#subdirectory=src'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'gradio==6.26.0', 'matplotlib>=3.8,<4'], check=True)
import luna_genai
assert luna_genai.__version__ == '0.1.14', '패키지 버전이 다릅니다. 설치 후 런타임을 재시작하세요.'
from luna_genai.review_core import definition_link
from luna_genai.review2_core import (reconstruction_stats, vae_quantities, cosine_search,
    precision_at_k, schedule_coefficients, diffusion_state, recover_clean,
    cfg_vector, ddim_step, load_review_records)
from luna_genai.review2_app import (representation_view, search_view, diffusion_view,
    sampling_view, recorded_view, build_review_app)
from IPython.display import display, Markdown, Image
import matplotlib.pyplot as plt
print('Python', sys.version.split()[0], '| course', luna_genai.__version__, COURSE_REF)
print('설명용 수치 계산 및 기존 기록 열람: 새로운 모델 학습/이미지 생성 없음')


### 설치한 구현의 정의 링크 확인

실제 파일의 시작 행을 읽어 같은 고정 태그 링크를 만듭니다. 설치 태그와 API 링크가 일치해야 합니다.


In [ ]:
for fn in [reconstruction_stats, vae_quantities, cosine_search, precision_at_k, representation_view, search_view]:
    display(Markdown(f"[{fn.__name__}]({definition_link(fn, COURSE_REF)})"))


## 1. 픽셀 비교와 VAE 계산

원본과 복원은 같은 픽셀 위치끼리 비교합니다. $\mathrm{SSE}=\sum_i(x_i-\hat x_i)^2$, $\mathrm{MSE}=\mathrm{SSE}/P$입니다. 아래는 한 이미지 네 픽셀의 설명용 계산이며 신경망을 훈련하지 않습니다.

VAE는 $\sigma=\exp(\ell/2)$와 $z=\mu+\sigma\odot\epsilon$을 구분합니다. $\ell$은 로그분산입니다. 고정된 복원과 잠재분포에 beta를 곱해 합을 바꾸는 일과 beta를 바꿔 모델을 재학습하는 일은 다릅니다.


In [ ]:
demo_pixels, demo_latent, demo_report, demo_figure = representation_view(1., -.4)
display(demo_pixels); display(demo_latent); display(Markdown(demo_report))
display(demo_figure); plt.close(demo_figure)


## 2. 검색에서 유지되는 값과 바뀌는 값

질의와 후보를 행별로 정규화한 뒤 내적을 계산합니다. `order`는 원래 후보 인덱스의 순위이며 동률은 원래 순서로 정합니다. 아래는 실제 CLIP을 실행한 임베딩이 아닌 작은 설명용 벡터입니다.

같은 기존 벡터에 후보만 추가하면 기존 cosine은 유지되지만 상대 점수의 분모가 바뀝니다. 관련성은 검색 결과를 보기 전에 A,C로 정했습니다. Precision@k의 분모는 k입니다.


In [ ]:
demo_without = cosine_search([2,1], [[1,0],[0,1],[1,1]], 3.)
demo_with = cosine_search([2,1], [[1,0],[0,1],[1,1],[-1,0]], 3.)
print('기존 cosine:', demo_without['cosine'], demo_with['cosine'][:3])
print('후보A 상대점수:', demo_without['relative_score'][0], demo_with['relative_score'][0])
demo_rows, demo_report, demo_figure = search_view(True, 3., 2)
display(demo_rows); display(Markdown(demo_report)); display(demo_figure); plt.close(demo_figure)


## Python 연습 · 본인의 풀이 기록

Notion·PDF와 같은 ID와 조건입니다. 실행 전에 예상하고, 수정 후 변수의 중간값을 확인하세요. 미완료 기록칸은 자동으로 채워지지 않습니다.


### Q07 · 로그분산을 표준편차로 바꾸는 빈칸

math.exp는 지수함수다. 아래 빈칸 하나를 채워 로그분산을 표준편차로 바꾸고 result를 구하라. ε는 제공된 값으로 고정하며 각 좌표를 따로 계산한다.

```python
import math
mu = [0.0, 1.0]
logvar = [math.log(4.0), math.log(0.25)]
epsilon = [-0.5, 2.0]
std = [math.exp(___ * value) for value in logvar]
result = [mu[i] + std[i] * epsilon[i] for i in range(2)]
print(result)
```

목표: 공식의 로그분산→표준편차 변환을 Python 원소별 연산에 연결한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q07_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q08 · 평균 제곱오차의 잘못된 분모

배치는 두 이미지, 이미지마다 픽셀 두 개다. 전체 네 픽셀의 평균 제곱오차를 구하려고 아래 코드를 작성했다. 잘못된 줄을 고치고 수정 전·후 값을 비교하라. target과 pred는 같은 범위의 픽셀값이다.

```python
target = [[0.0, 1.0], [0.5, 0.5]]
pred = [[0.2, 0.8], [0.5, 0.7]]
sse = sum((target[b][p] - pred[b][p]) ** 2
          for b in range(2) for p in range(2))
result = sse / len(target)
print(result)
```

목표: 배치 평균과 모든 원소 평균을 코드에서 구별한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q08_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q09 · 평균 복원과 잠재 샘플 추적

아래 코드는 신경망 성능을 재현하는 코드가 아니라, 평균 사용과 표본 사용의 차이를 보여 주는 작은 산술 예다. decode(z)=2z를 사용한다. 출력 두 줄을 구하고, 평균을 디코딩한 값과 두 잡음 표본에서의 값을 구별해 설명하라. 표본 두 개의 평균이 일반적으로 기대값과 정확히 같은지도 판단하라.

```python
mu, std = 1.5, 0.5
def decode(z):
    return 2 * z
mean_reconstruction = decode(mu)
sampled = [decode(mu + std * e) for e in [-1.0, 2.0]]
print(mean_reconstruction)
print(sampled)
```

목표: 평균으로 고정한 복원과 잠재 표본에 따른 출력을 구별한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q09_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q10 · VAE 손실의 두 항 연결

한 이미지의 복원 제곱오차 합과 제공된 KL로 총손실을 계산한다. 수업 정의는 이미지 안에서 복원항을 합하고, 그 뒤 β·KL을 더하는 것이다. 빈칸을 채우고 recon_sse, weighted_kl, total을 구하라. KL 공식의 유도는 요구하지 않는다.

```python
pixel_errors = [0.1, -0.2, 0.1]
kl = 0.4
beta = 0.5
recon_sse = sum(e * e for e in pixel_errors)
weighted_kl = ___ * kl
total = recon_sse + weighted_kl
print(recon_sse, weighted_kl, total)
```

목표: 복원과 분포 제약의 합을 코드에 연결하고 reduction의 영향을 해석한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q10_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q18 · 큰 점수 순서의 안정 정렬

scores[i]는 사진 i의 유사도다. sorted의 key에는 정렬할 값을 반환하는 함수를 넣으며 reverse=True는 큰 값부터 정렬한다. Python의 안정 정렬은 동점에서 원래 순서를 유지한다. 빈칸을 채워 상위3개 사진 ID를 구하라.

```python
scores = [0.4, 0.9, 0.9, 0.2]
ids = ['A', 'B', 'C', 'D']
order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=___)
result = [ids[i] for i in order[:3]]
print(result)
```

목표: 유사도 점수의 큰 순서와 사진 ID 대응을 유지한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q18_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q19 · 사진마다 길이를1로 만들기

E의 한 행이 사진 하나이고 두 열은 특징이다. 아래 코드는 모든 원소의 길이 하나로 행렬 전체를 나누었다. 사진마다 길이가1이 되도록 고치고 두 정규화 행을 써라. 모든 행은 영벡터가 아니며 math.sqrt는 제곱근이다.

```python
import math
E = [[0.0, 2.0], [3.0, 4.0]]
length = math.sqrt(sum(v*v for row in E for v in row))
normalized = [[v / length for v in row] for row in E]
print(normalized)
```

목표: 행렬 축 의미와 행별 정규화를 연결한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q19_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


### Q20 · 검색 방향을 복제한 코드 오류

S의 행은 이미지 I0,I1,I2, 열은 문장 T0,T1이다. max(indices,key=...)는 key 점수가 가장 큰 인덱스를 반환한다. 문장→이미지 검색을 만들면서 이미지→문장 코드를 그대로 복사했다. 잘못된 줄을 고치고 수정 전·후 두 출력의 의미와 길이를 설명하라. 점수 동점은 없다.

```python
S = [[0.8, 0.1], [0.2, 0.7], [0.4, 0.6]]
image_to_text = [max(range(2), key=lambda j: row[j]) for row in S]
text_to_image = [max(range(2), key=lambda j: row[j]) for row in S]
print(image_to_text)
print(text_to_image)
```

목표: 행/열의 검색 방향 오류를 고치고 출력 인덱스의 의미를 추적한다.

풀이 방법: 실행 전에 각 변수의 값과 길이를 적고, 수정 후 결과의 의미를 설명합니다. 아래 기록칸과 별도의 코드 셀에 직접 작성하세요.


In [ ]:
q20_my_work = None  # 미완료: 본인의 이유와 코드를 별도 셀에 작성하세요.


## 독립 실행 검사

아래 검사는 문제은행과 다른 입력의 계산 관계·데이터 무결성·callback 계약을 확인합니다. 연습문제의 답을 공개하거나 자신의 풀이를 자동 채점하지 않습니다.


In [ ]:
check_stats = reconstruction_stats([.15,.85],[.25,.75])
assert math.isclose(check_stats['sse'],2*check_stats['mse'])
check_vae = vae_quantities([0,0],[0,0],[.4,-.6])
assert check_vae['kl']==0 and check_vae['std']==[1.,1.]
check_search = cosine_search([1,2],[[1,2],[2,1]],2)
assert check_search['order']==[0,1] and math.isclose(sum(check_search['relative_score']),1)
assert precision_at_k(['X','Y'],{'Y'},2)==.5
check_rows,check_message,check_figure=search_view(False,2.,2)
assert len(check_rows)==3 and '설명용 수치 계산' in check_message
plt.close(check_figure)
print('W08A-A RUNTIME CHECK PASSED')


## 풀이 완료와 실행 성공은 별개

`qXX_my_work=None`은 본인의 풀이 기록이 아직 비어 있다는 표시입니다. 자신의 설명과 코드를 작성해도 자동 채점은 수행하지 않습니다. 기록 열람·계산 검사·문제 풀이 완료를 구별하여 복습하세요.


In [ ]:
print('아직 풀이 기록이 없는 문제:', [name for name,value in list(globals().items()) if name.startswith('q') and name.endswith('_my_work') and value is None])
